In [ ]:
# Step 5: Association Rule Mining

from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# --- Load transaction data ---
df_txn = pd.read_csv("../data/customer_transactions.csv")

# --- Convert to transactional format ---
basket = df_txn.groupby(['customer_id'])['product_category'].apply(list).reset_index()
transactions = basket['product_category'].tolist()

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# --- Apply Apriori Algorithm ---
display(Markdown("## 🔍 Applying Apriori Algorithm"))
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values(by="support", ascending=False)
display(frequent_itemsets.head(10))

# --- Generate Association Rules ---
display(Markdown("## 📊 Generating Association Rules"))
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules.sort_values(by='lift', ascending=False)
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

# --- Plot top rules ---
plt.figure(figsize=(10, 6))
plt.scatter(rules['support'], rules['confidence'], alpha=0.6, c=rules['lift'], cmap='viridis')
plt.title("Association Rules: Support vs Confidence")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.colorbar(label='Lift')
plt.tight_layout()
plt.show()

# --- Commentary ---
display(Markdown("""
### 📌 Interpretation:
- Rules with high **confidence** and **lift** are most actionable.
- This analysis helps discover co-purchased product categories.
- Can be used to inform product bundling or recommendation systems.
"""))
